# Lab 4: Geospatial Python Libraries - GeoPandas
#### Interactive Jupyter Notebook

This notebook provides an introduction to <b>GeoPandas</b>, a <b>Python geospatial library</b> that combines the tables of Pandas with the geometries of Shapely.

## Notebook Outline:
- [Introduction](#intro)
- [Check Libraries](#run_once)
- [Setup](#setup) (run this first!)
- [GeoPandas](#geoPandas)
    - [Import Data](#import_data)
    - [Reading File into a GeoDataFrame](#read_file)
    - [Explore DataFrame](#exploreDF)
    - [Sort a DataFrame](#sortDF)
    - [Selecting rows and columns in a data frame](#selectDF)
    - [Plotting GeoDataFrames](#plotDF)
    - [Spatial Operations](#spatialOP)
- [New Example problem](#newProb)
    - [Reading Data in GeoDataFrame](#readGDF)
    - [Plotting Data](#plotData)
    - [Choropleth Maps](#clMaps)
    - [Maps with Layers](#mapLayers)
    - [Aggregation of geometric features](#aggrGeom)
    - [Merging Data](#mergeData)
    - [Saving Data](#saveData)

<a id='intro'></a>
## Introduction

<b>GeoPandas</b> extends Pandas by adding core geographical functionality. It enables spatial operations on vector data types using Shapely, and reads and writes files using pyogrio or Fiona. You can do spatial database-like operations with this library.

Useful documentation for these libraries can be found here:

>GeoPandas https://geopandas.org/
<br>mapclassify (classification schemes for choropleth maps) https://pysal.org/mapclassify/

<a id='run_once'></a>
## Check Libraries
This cell checks that the libraries this notebook needs are installed, and installs any that are missing. If it installs anything, choose <b>Kernel -> Restart Kernel</b>, then continue with Setup.

In [ ]:
import importlib.util

missing = [p for p in ['geopandas', 'mapclassify']
           if importlib.util.find_spec(p) is None]
if missing:
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(missing))
    print('Installed:', ', '.join(missing), '- now choose Kernel -> Restart Kernel, then run Setup.')
else:
    print('All libraries are available.')

<a id='setup'></a>
## Setup
Run this cell for the rest of the notebook to work!

In [ ]:
# import required libraries

%matplotlib inline
import os

# use Shapely 2.0 instead of PyGEOS in GeoPandas (must come before importing geopandas)
os.environ['USE_PYGEOS'] = '0'

import geopandas as gpd
from geopandas import GeoSeries, GeoDataFrame, read_file
from pandas import Series
from matplotlib import pyplot as plt
import IPython

print('Setup complete.')

<a id='geoPandas'></a>
## GeoPandas

<a id='import_data'></a>
### Import Data
We will be using the New York City borough boundaries for parts of this lab. The data comes from the NYC Department of City Planning (also available on NYC Open Data) and is in the `data/nybb` folder next to this notebook.

<a id='read_file'></a>
### Reading File into a GeoDataFrame

Load the input_file using GeoPandas in Data Frame.

In [ ]:
input_file = 'data/nybb/nybb.shp'
if os.path.exists(input_file):
    print('Input file:', input_file)
else:
    print('Could not find the data. Make sure this notebook is in the ggis407-lab4 folder, next to the data folder.')

In [ ]:
boroDF=gpd.read_file(input_file)
boroDF

<a id='exploreDF'></a>
### Explore DataFrame

In [ ]:
# see index
boroDF.index

In [ ]:
# see column names
boroDF.columns

In [ ]:
# see values
boroDF.values

In [ ]:
# see the size of the values array
boroDF.values.shape

In [ ]:
# count columns
print (len(boroDF.values[0]))
print (len(boroDF.columns))

In [ ]:
# Using index selection we can see all values in one row.
boroDF.values[1]

In [ ]:
# use the double index method to find a value within a particular row and column
boroDF.values[1][1]

In [ ]:
# summarize DataFrame
boroDF.describe()

In [ ]:
# transpose DataFrame
boroDF.T

<a id='sortDF'></a>
### Sort a DataFrame

In [ ]:
boroDF.sort_values(by="boro_code",inplace=True,ascending=True)

In [ ]:
boroDF.head()

<a id="selectDF"></a>
### Selecting rows and columns in a data frame

In [ ]:
# Show the first 2 boroughs
boroDF[:2]

In [ ]:
# Sort again by area, then show two largest boroughs
boroDF.sort_values(by="shape_area",
                   inplace=True,
                   ascending=False)
boroDF[:2]

In [ ]:
# Select two of the columns
boroDF[['shape_area', 'boro_name']]

In [ ]:
# Select two named columns, and the first two rows
boroDF[['shape_area', 'boro_name']][:2]

In [ ]:
# Select two rows and then three named columns
boroDF[:2][['shape_area', 'boro_name','shape_leng']]

In [ ]:
# To add a column, first, get a list of the
# same length of the data frame
bpop = [2000000,2200000,600000,1100000,2600000]

In [ ]:
# Add a column of totally fake population numbers
boroDF["Population"] = bpop

In [ ]:
boroDF.tail()

In [ ]:
# Make a series that will join to the right boroughs
# (2020 Census population counts)
population = Series({'Manhattan': 1694251,
                     'Bronx': 1472654,
                     'Brooklyn': 2736074,
                     'Queens': 2405464,
                     'Staten Island': 495747})

In [ ]:
# If you do this step more than once it will fail since the column is already an index column
boroDF = boroDF.set_index("boro_name")

In [ ]:
# Add a column of real population numbers
boroDF["Population"] = population

In [ ]:
boroDF

The population values were matched to rows by borough name (the index), not by their order. This is called index alignment

<a id="plotDF"></a>
### Plotting GeoDataFrames
Plotting is much easier since they can be plotted directly; we don't have to harvest the coordinates from the data structure.

In [ ]:
boroDF.plot(figsize=(8, 8))

GeoPandas can also make an interactive web map with `explore()`, which you saw in Lab 3. Hover over a borough to see its attributes. `reset_index()` turns `boro_name` back into a regular column so it shows up in the tooltip.

In [ ]:
esri_url = 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}'
boroDF.reset_index().explore(column='Population', cmap='YlOrRd', tiles=esri_url, attr='Tiles &copy; Esri')

In [ ]:
boroDF

In [ ]:
# We can also get individual features using their position
# with .iloc. The table is sorted by area, largest first,
# so what position should we use for Queens?
queens_position = 0
queensGeo = boroDF.geometry.iloc[queens_position]
queensGeo

In [ ]:
# Sort by population and print DF
boroDF.sort_values(by="Population",inplace=True,ascending=True)
boroDF

In [ ]:
boroDF

In [ ]:
# Get the geometry for Queens again, this time by its label
# (boro_name is now the index)
queensGeo = boroDF.geometry['Queens']
queensGeo

<a id="spatialOP"></a>
### Spatial operations
GeoPandas depends on Shapely ( https://shapely.readthedocs.io/ ), which provides these operations. GeoPandas also depends on several other libraries ( https://geopandas.org/en/stable/getting_started/install.html#dependencies ), including pyogrio or Fiona, which handle input and output of files.

Our borough data is in latitude and longitude (EPSG:4326), so distances for these operations are in <b>degrees</b>. In New York, 0.018 degrees is roughly 1–1.25 miles (it depends on direction). In Module 5 and later you'll see how to project data so you can work in meters or feet instead. When you run a buffer or area calculation on latitude/longitude data, GeoPandas shows a warning saying the results are likely incorrect. That's expected here; it's reminding you that the units are degrees.

In [ ]:
# Each geometry object has spatial
# operators available, like the following:
IPython.display.display(
    "Convex hull",
    queensGeo.convex_hull,
    "Difference of convex hull and feature",
    queensGeo.convex_hull.difference(queensGeo),
    "Difference of envelope and convex hull",
    queensGeo.envelope.difference(queensGeo.convex_hull),
    "Approximate one mile buffer on Queens (0.018 degrees)",
    queensGeo.buffer(0.018),
    "Approximate one mile negative buffer on Queens",
    queensGeo.buffer(-0.018),
    "Simplified geometry for Queens (tolerance 0.005 degrees)",
    queensGeo.simplify(0.005))

In [ ]:
# A smaller tolerance keeps more detail
queensGeo.simplify(0.001)

In [ ]:
queensGeo.envelope

In [ ]:
boroDF.geometry.convex_hull.plot()

In [ ]:
# You can also plot the result of
# a spatial operation on all features
boroDF.geometry.buffer(0.018).plot(figsize=(8, 8))

A buffer of 0.018 degrees is only roughly a mile, and it’s a slightly different distance north–south than east–west. To buffer by an exact distance, reproject to a CRS that uses real units. EPSG:2263 (New York State Plane, Long Island) is the standard projected CRS for New York City, and its units are feet, so a one-mile buffer is 5,280 feet.

In [ ]:
# Buffer every borough by exactly one mile (5,280 feet)
boroDF.to_crs(epsg=2263).geometry.buffer(5280).plot(figsize=(8, 8))

<a id="newProb"></a>
## Another Example problem
Next we use two small sample datasets from <a href="https://www.naturalearthdata.com/">Natural Earth</a>: country boundaries and major cities. (Data are in the `data` folder.)

<a id="readGDF"></a>
### Reading Data in GeoDataFrame

In [ ]:
# Load the sample datasets
cities = gpd.read_file('data/naturalearth_cities/naturalearth_cities.shp')

In [ ]:
world = gpd.read_file('data/naturalearth_lowres/naturalearth_lowres.shp')

In [ ]:
# Examine the world GeoDataFrame
world.head()

In [ ]:
cities.head()

In [ ]:
world.geom_type

In [ ]:
world.geometry

In [ ]:
# Area of each country, in square degrees.
# GeoPandas warns that this isn't a true area,
# because the data is in latitude/longitude.
world.geometry.area

Those numbers are in square degrees, which aren’t a real unit of area: a degree of longitude covers less ground the farther you are from the equator. To get real areas, first reproject the data to an equal-area coordinate reference system. Here we use EPSG:6933, whose units are meters, and divide by 1,000,000 to get square kilometers. The warning goes away because the data is no longer in latitude/longitude.

In [ ]:
# Area of each country in square kilometers
world_area_km2 = world.to_crs(epsg=6933).geometry.area / 1e6
world_area_km2

Let us make it a little bit more readable

In [ ]:
# Largest 5 countries by area
world.assign(area_km2=world_area_km2).sort_values('area_km2', ascending=False)[['name', 'area_km2']].head()

In [ ]:
print("There are", len(world),
      "countries in the sample dataset.")

In [ ]:
print("There are", len(cities),
      "cities in the sample dataset. Here are the first five:")
cities[:5]

<a id="plotData"></a>
### Plotting Data

In [ ]:
world.plot()

In [ ]:
cities.plot(figsize=(10, 10))

<a id="clMaps"></a>
### Choropleth Maps

In [ ]:
# Make choropleth maps based on world population
world.plot(column='pop_est')

In [ ]:
# Choosing color. Full list of colors on matplotlib
# website http://matplotlib.org/users/colormaps.html
world.plot(figsize=(14, 7), column='pop_est', 
           cmap='seismic',legend=True)

In [ ]:
# By default the colors are spread over a continuous
# range, but with mapclassify you can group values into
# classes with a scheme: 'quantiles', 'fisher_jenks', etc.
world.plot(figsize=(10, 5), column='pop_est',
           cmap='seismic', scheme='quantiles',
           legend=True);

In [ ]:
# Make choropleth maps based on GDP per person
# (gdp_md_est is in millions of US dollars)
world = world[(world.pop_est>0) & (world.name!="Antarctica")].copy()
world['gdp_per_cap'] = world.gdp_md_est / world.pop_est
world.plot(figsize=(10, 5), column='gdp_per_cap',
           legend=True);

In [ ]:
world.plot(column='gdp_md_est',
          legend=True);

In [ ]:
world.plot(figsize=(12, 6), column='gdp_md_est',
          cmap='OrRd', legend=True);

In [ ]:
# Group the values into classes using the
# Fisher-Jenks natural breaks scheme
world.plot(figsize=(12, 6), column='gdp_md_est',
           cmap='OrRd', legend=True,
           scheme='fisher_jenks');

In [ ]:
cities.plot(figsize=(8, 4), marker='*', color='green', 
            markersize=15);

<a id="mapLayers"></a>
### Maps with Layers
Combining multiple layers on the map. Remember to always ensure they share a common CRS (coordinate reference system), so they will align.

In [ ]:
world.crs

In [ ]:
cities.crs

In [ ]:
# Plotting Method 1
base = world.plot(color='white', edgecolor='black',figsize=(20, 10))
cities.plot(ax=base, marker='o', color='red',markersize=5)

In [ ]:
# Plotting using matplotlib objects
fig, ax = plt.subplots(figsize=(10,5))
# set aspect to equal. This is done automatically
# when using *geopandas* plot on its own,
# but not when working with pyplot directly.
ax.set_aspect('equal')
world.plot(ax=ax, color='white',
           edgecolor='black', linestyle=":")
cities.plot(ax=ax, marker='o', color='red',
            markersize=3);

#### Reprojecting

In [ ]:
# Reprojecting from WGS84 (latitude/longitude) to Mercator.
# Antarctica is already removed from world: the Mercator
# projection can't show the poles.
ax = world.plot(figsize=(20,10))
cities.plot(ax=ax, marker='o', color='red',
            markersize=10)
ax.set_title("WGS84 (lat/lon)");
ax = world.to_crs(epsg=3395).plot(figsize=(20,10))
cities.to_crs(epsg=3395).plot(ax=ax, marker='o', color='red',
                              markersize=10)
ax.set_title("World Mercator (EPSG:3395)");

<a id="aggrGeom"></a>
### Aggregation of geometric features
This is accomplished using the `dissolve` function. `dissolve` can be thought of as doing three things: (a) it dissolves all the geometries within a given group together into a single geometric feature (a union of the shapes); (b) it aggregates all the rows of data in a group using `groupby.aggregate()`; and (c) it combines those two results.

In [ ]:
# Suppose we want to aggregate data
# at continent level
world = gpd.read_file('data/naturalearth_lowres/naturalearth_lowres.shp')
world = world[['continent', 'geometry']]

In [ ]:
continents = world.dissolve(by='continent')
continents.plot(figsize=(10,5))

In [ ]:
# For looking at aggregate population of each continent
world = gpd.read_file('data/naturalearth_lowres/naturalearth_lowres.shp')
world = world[['continent',
               'geometry', 'pop_est']]
continents = world.dissolve(by='continent', aggfunc='sum')
continents.plot(column='pop_est', scheme='quantiles',
                cmap='YlOrRd', figsize=(10,5), legend=True);

<a id="mergeData"></a>
### Merging Data
There are two ways to combine datasets in geopandas – attribute joins and spatial joins.

In [ ]:
world = gpd.read_file('data/naturalearth_lowres/naturalearth_lowres.shp')
cities = gpd.read_file('data/naturalearth_cities/naturalearth_cities.shp')
# For attribute join
country_shapes = world[['geometry', 'iso_a3']]
country_names = world[['name', 'iso_a3']]
# For spatial join
countries = world[['geometry', 'name']]
countries = countries.rename(
    columns={'name':'country'})

In [ ]:
countries.head()

#### Attribute Join

In [ ]:
country_shapes.head()

In [ ]:
country_names.head()

In [ ]:
# Tabular join using 'merge' method
country_shapes = country_shapes.merge(
    country_names, on='iso_a3')
country_shapes.head()

#### Spatial Join

In [ ]:
countries.head()

In [ ]:
cities.head()

In [ ]:
# Spatial join. The 'predicate' option determines
# the type of join operation to apply.
# predicate can be set to "intersects", "within" or
# "contains" (these are all equivalent when
# joining points to polygons, but differ when
# joining polygons to other polygons or lines).
cities_with_country = gpd.sjoin(
    cities, countries, predicate='intersects')

In [ ]:
cities_with_country.head()

Notice that `cities_with_country` has fewer rows than `cities`. A few cities fall just outside the simplified country outlines (for example, on small islands or the coast), so `intersects` doesn't match them to any country:

In [ ]:
print("Cities:", len(cities), "| Cities matched to a country:", len(cities_with_country))

Now that each city has a country, we can use what you learned about Pandas to summarize the result, for example by counting the cities in each country:

In [ ]:
# number of cities in each country
cities_with_country.groupby('country').size().sort_values(ascending=False).head(10)

<a id="saveData"></a>
### Saving Data
GeoPandas can write a GeoDataFrame to a file with `to_file`. A GeoPackage (`.gpkg`) keeps everything in one file, unlike a shapefile's several, and has no 10-character limit on column names. The file is saved in the same folder as this notebook.

In [ ]:
# save the result to a GeoPackage file (one file, unlike a shapefile's several)
cities_with_country.to_file('cities_with_country.gpkg', driver='GPKG')

In [ ]:
# Practice what you have learned about GeoPandas
# Feel free to modify any code above
# Try these: 
#     Use cities_with_country to count the cities on each continent. (Hint: you’ll need the continent column from world in your spatial join.)
#     Make a choropleth of GDP per person using the quantiles scheme. How does it compare with the default map?